# 如何使用 LangChain 构建使用工具的智能体

本 notebook 将带你了解如何使用 LangChain 为 OpenAI 模型接入外部工具。具体来说，你将能够创建使用自定义工具来回答用户查询的 LLM 智能体（agent）。


## 什么是 Langchain？
[LangChain](https://python.langchain.com/en/latest/index.html) 是一个用于开发由语言模型驱动的应用的框架。借助该框架，你可以构建分层的、由 LLM 驱动的应用，这些应用具备上下文感知能力，并能作为智能体与其环境动态交互，从而为你简化代码，并为你的客户带来更动态的用户体验。

## 为什么 LLM 需要使用工具？
LLM 最常见的挑战之一，是如何克服其训练数据在时效性和针对性上的不足——答案可能已经过时，而且由于其知识库内容极其庞杂，模型容易产生幻觉。工具是一种很好的方法，能让 LLM 在受控的上下文中作答，并利用你现有的知识库和内部 API——与其费尽心思通过提示工程把 LLM 一路引导到你想要的答案，不如让它访问工具，动态调用这些工具获取信息、进行解析，并提供给客户。 

为 LLM 提供工具访问能力，可以让它们直接利用来自搜索引擎、API 或你自己数据库的上下文来回答问题。拥有工具访问能力的 LLM 不会直接作答，而是可以执行中间步骤来收集相关信息。工具也可以组合使用。[例如](https://python.langchain.com/en/latest/modules/agents/agents/examples/mrkl_chat.html)，可以让语言模型使用搜索工具查找定量信息，再用计算器执行计算。

## Notebook 章节

- **环境设置：** 导入软件包并连接 Pinecone 向量数据库。
- **LLM 智能体：** 构建一个利用修改版 [ReAct](https://react-lm.github.io/) 框架进行思维链推理的智能体。
- **带历史记录的 LLM 智能体：** 让 LLM 能够访问对话中之前的步骤。
- **知识库：** 创建一个由“Stuff You Should Know”播客节目构成的知识库，通过工具进行访问。
- **带工具的 LLM 智能体：** 为智能体扩展多个工具的访问能力，并测试它是否会使用这些工具来回答问题。

In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 环境设置

导入库并建立与 [Pinecone](https://www.pinecone.io) 向量数据库的连接。

你可以将 Pinecone 替换为任何其他向量存储或数据库——Langchain 原生支持[一系列](https://python.langchain.com/en/latest/modules/indexes/vectorstores.html)向量存储，其他连接器则需要你自行开发。

In [ ]:
!pip install openai
!pip install pinecone-client
!pip install pandas
!pip install typing
!pip install tqdm
!pip install langchain
!pip install wget

In [ ]:
import datetime
import json
import openai
import os
import pandas as pd
import pinecone
import re
from tqdm.auto import tqdm
from typing import List, Union
import zipfile

# Langchain imports
from langchain.agents import Tool, AgentExecutor, LLMSingleActionAgent, AgentOutputParser
from langchain.prompts import BaseChatPromptTemplate, ChatPromptTemplate
from langchain import SerpAPIWrapper, LLMChain
from langchain.schema import AgentAction, AgentFinish, HumanMessage, SystemMessage
# LLM wrapper
from langchain.chat_models import ChatOpenAI
from langchain import OpenAI
# Conversational memory
from langchain.memory import ConversationBufferWindowMemory
# Embeddings and vectorstore
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Pinecone

# Vectorstore Index
index_name = 'podcasts'

要获取用于连接 Pinecone 的 API 密钥，你可以注册一个[免费账户](https://app.pinecone.io/)，并将密钥存储在下方的 `api_key` 变量中，或存储在环境变量 `PINECONE_API_KEY` 中

In [4]:
api_key = os.getenv("PINECONE_API_KEY") or "PINECONE_API_KEY"

# find environment next to your API key in the Pinecone console
env = os.getenv("PINECONE_ENVIRONMENT") or "PINECONE_ENVIRONMENT"

pinecone.init(api_key=api_key, environment=env)
pinecone.whoami()

In [5]:
pinecone.list_indexes()

['podcasts']

如果你想清空索引，或者索引尚不存在，请运行此代码块

```
# Check whether the index with the same name already exists - if so, delete it
if index_name in pinecone.list_indexes():
    pinecone.delete_index(index_name)
    
# Creates new index
pinecone.create_index(name=index_name, dimension=1536)
index = pinecone.Index(index_name=index_name)

# Confirm our index was created
pinecone.list_indexes()
```

## LLM 智能体

Langchain 中的 [LLM 智能体](https://python.langchain.com/docs/modules/agents/) 有许多可配置的组件，Langchain 文档中对此有详细介绍。

我们将运用其中几个核心概念，构建一个以我们期望的方式对话、能使用工具回答问题、并使用合适的语言模型驱动对话的智能体。
- **提示词模板（Prompt Template）：** 用于控制 LLM 行为以及其接收输入和产生输出方式的输入模板——这是驱动你应用的大脑（[文档](https://python.langchain.com/en/latest/modules/prompts/prompt_templates.html)）。
- **输出解析器（Output Parser）：** 一种解析提示词输出的方法。如果 LLM 使用特定的标题格式产生输出，你就可以实现复杂的交互：由 LLM 在响应中生成变量，并将其传入链的下一步（[文档](https://python.langchain.com/en/latest/modules/prompts/output_parsers.html)）。
- **LLM 链（LLM Chain）：** 链将提示词模板与执行它的 LLM 组合在一起——本例中我们使用 ```gpt-3.5-turbo```，但该框架也可以与 OpenAI 补全模型或完全不同的其他 LLM 一起使用（[文档](https://python.langchain.com/en/latest/modules/chains.html)）。
- **工具（Tool）：** 一种外部服务，当用户需要时，LLM 可以用它来获取信息或执行命令（[文档](https://python.langchain.com/en/latest/modules/agents/tools.html)）。
- **智能体（Agent）：** 将上述一切粘合在一起的部分，智能体可以调用多个 LLM 链，每个链都有自己的工具。你可以用自己的逻辑扩展智能体，加入重试、错误处理以及任何你选择的方法，以提高应用的可靠性（[文档](https://python.langchain.com/en/latest/modules/agents.html)）。

**注意：** 在将本 cookbook 与 Search 工具一起使用之前，你需要在 https://serpapi.com/ 注册并生成一个 API 密钥。获得密钥后，将其存储在名为 ```SERPAPI_API_KEY``` 的环境变量中

In [34]:
# Initiate a Search tool - note you'll need to have set SERPAPI_API_KEY as an environment variable as per the above instructions
search = SerpAPIWrapper()

# Define a list of tools
tools = [
    Tool(
        name = "Search",
        func=search.run,
        description="useful for when you need to answer questions about current events"
    )
]

In [35]:
# Set up the prompt with input variables for tools, user input and a scratchpad for the model to record its workings
template = """Answer the following questions as best you can, but speaking as a pirate might speak. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin! Remember to speak as a pirate when giving your final answer. Use lots of "Arg"s

Question: {input}
{agent_scratchpad}"""

In [36]:
# Set up a prompt template
class CustomPromptTemplate(BaseChatPromptTemplate):
    # The template to use
    template: str
    # The list of tools available
    tools: List[Tool]
    
    def format_messages(self, **kwargs) -> str:
        # Get the intermediate steps (AgentAction, Observation tuples)
        
        # Format them in a particular way
        intermediate_steps = kwargs.pop("intermediate_steps")
        thoughts = ""
        for action, observation in intermediate_steps:
            thoughts += action.log
            thoughts += f"\nObservation: {observation}\nThought: "
            
        # Set the agent_scratchpad variable to that value
        kwargs["agent_scratchpad"] = thoughts
        
        # Create a tools variable from the list of tools provided
        kwargs["tools"] = "\n".join([f"{tool.name}: {tool.description}" for tool in self.tools])
        
        # Create a list of tool names for the tools provided
        kwargs["tool_names"] = ", ".join([tool.name for tool in self.tools])
        formatted = self.template.format(**kwargs)
        return [HumanMessage(content=formatted)]
    
prompt = CustomPromptTemplate(
    template=template,
    tools=tools,
    # This omits the `agent_scratchpad`, `tools`, and `tool_names` variables because those are generated dynamically
    # This includes the `intermediate_steps` variable because that is needed
    input_variables=["input", "intermediate_steps"]
)

In [37]:
class CustomOutputParser(AgentOutputParser):
    
    def parse(self, llm_output: str) -> Union[AgentAction, AgentFinish]:
        
        # Check if agent should finish
        if "Final Answer:" in llm_output:
            return AgentFinish(
                # Return values is generally always a dictionary with a single `output` key
                # It is not recommended to try anything else at the moment :)
                return_values={"output": llm_output.split("Final Answer:")[-1].strip()},
                log=llm_output,
            )
        
        # Parse out the action and action input
        regex = r"Action: (.*?)[\n]*Action Input:[\s]*(.*)"
        match = re.search(regex, llm_output, re.DOTALL)
        
        # If it can't parse the output it raises an error
        # You can add your own logic here to handle errors in a different way i.e. pass to a human, give a canned response
        if not match:
            raise ValueError(f"Could not parse LLM output: `{llm_output}`")
        action = match.group(1).strip()
        action_input = match.group(2)
        
        # Return the action and action input
        return AgentAction(tool=action, tool_input=action_input.strip(" ").strip('"'), log=llm_output)
    
output_parser = CustomOutputParser()

In [38]:
# Initiate our LLM - default is 'gpt-3.5-turbo'
llm = ChatOpenAI(temperature=0)

# LLM chain consisting of the LLM and a prompt
llm_chain = LLMChain(llm=llm, prompt=prompt)

# Using tools, the LLM chain and output_parser to make an agent
tool_names = [tool.name for tool in tools]

agent = LLMSingleActionAgent(
    llm_chain=llm_chain, 
    output_parser=output_parser,
    # We use "Observation" as our stop sequence so it will stop when it receives Tool output
    # If you change your prompt template you'll need to adjust this as well
    stop=["\nObservation:"], 
    allowed_tools=tool_names
)

In [39]:
# Initiate the agent that will respond to our queries
# Set verbose=True to share the CoT reasoning the LLM goes through
agent_executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)

In [40]:
agent_executor.run("How many people live in canada as of 2023?")



> Entering new AgentExecutor chain...
Thought: Hmm, I be not sure of the answer to that one. Let me think.
Action: Search
Action Input: "Canada population 2023"

Observation:39,566,248Ahoy, that be a lot of people! But I need to make sure this be true.
Action: Search
Action Input: "Canada population 2023 official source"

Observation:The current population of Canada is 38,664,637 as of Wednesday, April 19, 2023, based on Worldometer elaboration of the latest United Nations data.Arrr, that be the official number! I be confident in me answer now.
Final Answer: The population of Canada as of 2023 is 38,664,637. Arg!

> Finished chain.


'The population of Canada as of 2023 is 38,664,637. Arg!'

In [41]:
agent_executor.run("How many in 2022?")



> Entering new AgentExecutor chain...
Thought: Hmm, I'm not sure what this question is asking about. I better use the search tool.
Action: Search
Action Input: "2022 events"

Observation:8. Humanitarian Crises Deepen · 7. Latin America Moves Left. · 6. Iranians Protest. · 5. COVID Eases. · 4. Inflation Returns. · 3. Climate Change ...Ahoy, it looks like this be a question about what be happenin' in 2022. Let me search again.
Action: Search
Action Input: "2022 calendar"

Observation:United States 2022 – Calendar with American holidays. Yearly calendar showing months for the year 2022. Calendars – online and print friendly – for any year ...Shiver me timbers, it looks like this be a question about the year 2022. Let me search one more time.
Action: Search
Action Input: "What be happenin' in 2022?"

Observation:8. Humanitarian Crises Deepen · 7. Latin America Moves Left. · 6. Iranians Protest. · 5. COVID Eases. · 4. Inflation Returns. · 3. Climate Change ...Avast ye, it looks like the s

"Arg, I be sorry matey, but I can't give ye a clear answer to that question."

## 带历史记录的 LLM 智能体

为 LLM 智能体扩展保留[记忆](https://python.langchain.com/en/latest/modules/agents/agents/custom_llm_agent.html#adding-memory)的能力，并在继续对话时将其用作上下文。

本例中我们使用一个简单的 ```ConversationBufferWindowMemory```，它会保留最近两轮对话的滚动窗口。LangChain 还提供其他[记忆选项](https://python.langchain.com/en/latest/modules/memory.html)，各有不同的取舍，适用于不同的用例。

In [1]:
# Set up a prompt template which can interpolate the history
template_with_history = """You are SearchGPT, a professional search engine who provides informative answers to users. Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin! Remember to give detailed, informative answers

Previous conversation history:
{history}

New question: {input}
{agent_scratchpad}"""

In [43]:
prompt_with_history = CustomPromptTemplate(
    template=template_with_history,
    tools=tools,
    # The history template includes "history" as an input variable so we can interpolate it into the prompt
    input_variables=["input", "intermediate_steps", "history"]
)

llm_chain = LLMChain(llm=llm, prompt=prompt_with_history)
tool_names = [tool.name for tool in tools]
agent = LLMSingleActionAgent(
    llm_chain=llm_chain, 
    output_parser=output_parser,
    stop=["\nObservation:"], 
    allowed_tools=tool_names
)

In [44]:
# Initiate the memory with k=2 to keep the last two turns
# Provide the memory to the agent
memory = ConversationBufferWindowMemory(k=2)
agent_executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True, memory=memory)

In [45]:
agent_executor.run("How many people live in canada as of 2023?")



> Entering new AgentExecutor chain...
Thought: I need to find the most recent population data for Canada.
Action: Search
Action Input: "Canada population 2023"

Observation:39,566,248This data seems reliable, but I should double-check the source.
Action: Search
Action Input: "Source of Canada population 2023"

Observation:The current population of Canada is 38,664,637 as of Wednesday, April 19, 2023, based on Worldometer elaboration of the latest United Nations data. Canada 2020 population is estimated at 37,742,154 people at mid year according to UN data. Canada population is equivalent to 0.48% of the total world population.I now know the final answer
Final Answer: As of April 19, 2023, the population of Canada is 38,664,637.

> Finished chain.


'As of April 19, 2023, the population of Canada is 38,664,637.'

In [46]:
agent_executor.run("how about in mexico?")



> Entering new AgentExecutor chain...
Thought: I need to search for the current population of Mexico.
Action: Search
Action Input: "current population of Mexico"

Observation:Mexico, officially the United Mexican States, is a country in the southern portion of North America. It is bordered to the north by the United States; to the south and west by the Pacific Ocean; to the southeast by Guatemala, Belize, and the Caribbean Sea; and to the east by the Gulf of Mexico.That's not the answer to the question, I need to refine my search.
Action: Search
Action Input: "population of Mexico 2023"

Observation:132,709,512I now know the final answer.
Final Answer: As of 2023, the population of Mexico is 132,709,512.

> Finished chain.


'As of 2023, the population of Mexico is 132,709,512.'

## 知识库

创建一个自定义向量存储，供智能体作为工具来回答问题。我们将把结果存储在 LangChain 支持的 [Pinecone](https://docs.pinecone.io/docs/quickstart) 中（[文档](https://python.langchain.com/en/latest/modules/indexes/vectorstores/examples/pinecone.html)、[API 参考](https://python.langchain.com/en/latest/reference/modules/vectorstore.html)）。如果你需要 Pinecone 或其他向量数据库的入门帮助，我们有一个 [cookbook](https://github.com/openai/openai-cookbook/blob/colin/examples/vector_databases/Using_vector_databases_for_embeddings_search.ipynb) 可以帮你上手。

你可以查阅 LangChain 文档，了解还有哪些其他可用的[向量存储](https://python.langchain.com/en/latest/modules/indexes/vectorstores.html)和[数据库](https://python.langchain.com/en/latest/modules/chains/examples/sqlite.html)。

本例中我们将使用 Stuff You Should Know 播客的文字稿，感谢 OSF DOI [10.17605/OSF.IO/VM9NT](https://doi.org/10.17605/OSF.IO/VM9NT) 提供这些数据

In [18]:
import wget

# Here is a URL to a zip archive containing the transcribed podcasts
# Note that this data has already been split into chunks and embeddings from OpenAI's `text-embedding-3-small` embedding model are included
content_url = 'https://cdn.openai.com/API/examples/data/sysk_podcast_transcripts_embedded.json.zip'

# Download the file (it is ~541 MB so this will take some time)
wget.download(content_url)

100% [......................................................................] 571275039 / 571275039

'sysk_podcast_transcripts_embedded.json.zip'

In [19]:
# Load podcasts
with zipfile.ZipFile("sysk_podcast_transcripts_embedded.json.zip","r") as zip_ref:
    zip_ref.extractall("./data")
f = open('./data/sysk_podcast_transcripts_embedded.json')
processed_podcasts = json.load(f)

In [47]:
# Have a look at the contents
pd.DataFrame(processed_podcasts).head()

,id,filename,title,url,text_chunk,embedding,cleaned_id
0,sysk_with_transcripts_SYSK Selects How Crime S...,sysk_with_transcripts_SYSK Selects How Crime S...,\n\nSYSK Selects How Crime Scene Cleanup Works,https://chtbl.com/track/5899E/podtrac.com/pts/...,Title: sysk_with_transcripts_SYSK Selects How ...,"[0.021279960870742798, -0.005817972123622894, ...",sysk_with_transcripts_SYSK Selects How Crime S...
1,sysk_with_transcripts_SYSK Selects How Crime S...,sysk_with_transcripts_SYSK Selects How Crime S...,\n\nSYSK Selects How Crime Scene Cleanup Works,https://chtbl.com/track/5899E/podtrac.com/pts/...,Title: sysk_with_transcripts_SYSK Selects How ...,"[0.013859338127076626, 0.00857278611510992, 0....",sysk_with_transcripts_SYSK Selects How Crime S...
2,sysk_with_transcripts_SYSK Selects How Crime S...,sysk_with_transcripts_SYSK Selects How Crime S...,\n\nSYSK Selects How Crime Scene Cleanup Works,https://chtbl.com/track/5899E/podtrac.com/pts/...,Title: sysk_with_transcripts_SYSK Selects How ...,"[0.015242221765220165, 0.016030369326472282, 0...",sysk_with_transcripts_SYSK Selects How Crime S...
3,sysk_with_transcripts_SYSK Selects How Crime S...,sysk_with_transcripts_SYSK Selects How Crime S...,\n\nSYSK Selects How Crime Scene Cleanup Works,https://chtbl.com/track/5899E/podtrac.com/pts/...,Title: sysk_with_transcripts_SYSK Selects How ...,"[0.004371842369437218, -0.003036574460566044, ...",sysk_with_transcripts_SYSK Selects How Crime S...
4,sysk_with_transcripts_SYSK Selects How Crime S...,sysk_with_transcripts_SYSK Selects How Crime S...,\n\nSYSK Selects How Crime Scene Cleanup Works,https://chtbl.com/track/5899E/podtrac.com/pts/...,Title: sysk_with_transcripts_SYSK Selects How ...,"[0.017309172078967094, 0.015154214575886726, 0...",sysk_with_transcripts_SYSK Selects How Crime S...


In [ ]:
# Add the text embeddings to Pinecone

batch_size = 100  # how many embeddings we create and insert at once

for i in tqdm(range(0, len(processed_podcasts), batch_size)):
    # find end of batch
    i_end = min(len(processed_podcasts), i+batch_size)
    meta_batch = processed_podcasts[i:i_end]
    # get ids
    ids_batch = [x['cleaned_id'] for x in meta_batch]
    # get texts to encode
    texts = [x['text_chunk'] for x in meta_batch]
    # add embeddings
    embeds = [x['embedding'] for x in meta_batch]
    # cleanup metadata
    meta_batch = [{
        'filename': x['filename'],
        'title': x['title'],
        'text_chunk': x['text_chunk'],
        'url': x['url']
    } for x in meta_batch]
    to_upsert = list(zip(ids_batch, embeds, meta_batch))
    # upsert to Pinecone
    index.upsert(vectors=to_upsert)

In [48]:
# Configuring the embeddings to be used by our retriever to be OpenAI Embeddings, matching our embedded corpus
embeddings = OpenAIEmbeddings()


# Loads a docsearch object from an existing Pinecone index so we can retrieve from it
docsearch = Pinecone.from_existing_index(index_name,embeddings,text_key='text_chunk')

In [49]:
retriever = docsearch.as_retriever()

In [50]:
query_docs = retriever.get_relevant_documents("can you live without a bank account")

In [51]:
# Print out the title and content for the most relevant retrieved documents
print("\n".join(['Title: ' + x.metadata['title'].strip() + '\n\n' + x.page_content + '\n\n' for x in query_docs]))

Title: sysk: Can You Live Without a Bank Account?

Title: sysk_with_transcripts_Can you live without a bank account.json;  And if you had a life, you didn't necessarily rectify your bank checkbook every day. Oh, wait, what is balancing a checkbook mean? Seriously? Yeah. Thank God for my wife. So another reason you might avoid a bank is philosophically. There may be a longstanding distrust of banks in your family that you don't want to put your money in, or you may just want to be like, you know what? I don't want to take part in this modern society. I want to kind of drop out a bit. And a really good first move is to shut your bank account down. That's a big statement. Oh, yeah, it is. But a lot of people that are underbanked and don't have accounts aren't there on purpose. It's not some philosophical statement. A lot of times it's simply because they are poor and they don't have a lot of alternatives. Yeah. And the other thing about not having a bank account, not only do you not have 

## 带工具的 LLM 智能体

基于我们的 Pinecone 知识库创建一个 [RetrievalQA](https://python.langchain.com/en/latest/modules/chains/index_examples/vector_db_qa.html) 链，以扩展我们的工具列表。

In [52]:
from langchain.chains import RetrievalQA

retrieval_llm = OpenAI(temperature=0)

podcast_retriever = RetrievalQA.from_chain_type(llm=retrieval_llm, chain_type="stuff", retriever=docsearch.as_retriever())

In [53]:
expanded_tools = [
    Tool(
        name = "Search",
        func=search.run,
        description="useful for when you need to answer questions about current events"
    ),
    Tool(
        name = 'Knowledge Base',
        func=podcast_retriever.run,
        description="Useful for general questions about how to do things and for details on interesting topics. Input should be a fully formed question."
    )
]

In [54]:
# Re-initialize the agent with our new list of tools
prompt_with_history = CustomPromptTemplate(
    template=template_with_history,
    tools=expanded_tools,
    input_variables=["input", "intermediate_steps", "history"]
)
llm_chain = LLMChain(llm=llm, prompt=prompt_with_history)
multi_tool_names = [tool.name for tool in expanded_tools]
multi_tool_agent = LLMSingleActionAgent(
    llm_chain=llm_chain, 
    output_parser=output_parser,
    stop=["\nObservation:"], 
    allowed_tools=multi_tool_names
)

In [55]:
multi_tool_memory = ConversationBufferWindowMemory(k=2)
multi_tool_executor = AgentExecutor.from_agent_and_tools(agent=multi_tool_agent, tools=expanded_tools, verbose=True, memory=multi_tool_memory)

In [56]:
multi_tool_executor.run("Hi, I'd like to know how you can live without a bank account")



> Entering new AgentExecutor chain...
Thought: This is an interesting question. I'm not sure if I have the answer in my knowledge base, so I might need to search for it.
Action: Search
Action Input: "How to live without a bank account"

Observation:Underbanked households have a checking or savings account but also use alternative financial services such as money orders, check cashing, international remittances, payday loans, refund anticipation loans, rent-to-own services, pawnshop loans, or auto title loans, according to the FDIC.It seems like there are alternative financial services available for those who don't have a bank account. I should look into this further to provide a more comprehensive answer.
Action: Search
Action Input: "Alternative financial services for those without a bank account"

Observation:Instead, people who are unbanked use alternative financial services—payday loans, money orders, check cashing services, pawnshop loans, and the like—to meet their banking need

"While it is possible to live without a bank account by using alternative financial services, it may come with potential drawbacks and limitations. It's important to do research and compare options before making a decision, and there are resources available for those who may be interested in opening a bank account or exploring alternative financial services."

In [57]:
multi_tool_executor.run('Can you tell me some interesting facts about whether zoos are good or bad for animals')



> Entering new AgentExecutor chain...
Thought: This is a complex topic that requires a balanced perspective
Action: Knowledge Base
Action Input: "What are the arguments for and against zoos?"

Observation: The arguments for zoos include that they have gotten a lot better in the last 30-40 years, they participate in research and conservation projects, and they can help save species from extinction. The arguments against zoos include that they are still businesses, they can be counterproductive in terms of educating the public, and they can have a negative impact on the life span of animals in captivity.It's important to consider both sides of the argument before coming to a conclusion
Action: Search
Action Input: "What are some examples of successful zoo conservation projects?"

Observation:There are dedicated species survival programs which have helped species come out from the brink of extinction, good examples of that being the black-footed ferrets, the red wolves, the Przewalski's

"Zoos can have both positive and negative effects on animals, but they can play a role in conservation efforts for endangered species. It's important to consider both sides of the argument and do research before forming an opinion."

现在你已经有了一个部署带工具的对话式智能体的模板。如果你想通过自定义智能体（Custom Agent）进一步扩展，加入你自己的重试行为或对输入/输出变量的处理，请参阅[这篇文章](https://python.langchain.com/en/latest/modules/agents/agents/custom_agent.html)。

期待看到你的作品！